## Configuration Generator for Snowflake
This notebook facilitates the creation of one or multiple configurations in a single run. Follow the guidelines given below.

Initially, you should execute the entire notebook to make the widgets visible. Keep filling in the necessary information in the widgets and rerun the notebook until all widgets are populated.

#### Input Reference:
1. **Schema Selection**: Once you provide a connection string, this option will become available. Changing the schema will automatically update the dropdown list for table selection.
2. **Table Selection**: This accepts the same expressions as an `SQL LIKE` statement (e.g., `PPS%`). Use `%` to select all tables in a schema (exercise caution). For partial matches, use: `PPS%`. Note: Even if a table doesn't appear in the dropdown, you can still enter its name as we allow free form input. However, the dropdown is capped at 1024 items, so if your schema has more tables, you will have to input them manually.
3. **Destination for Configurations**: Choose the target location for your configuration files. Ensure that the designated configuration folder is created before you start the generation process.
4. **Bronze Load Type**: Here, you can select from the available load types (currently only overwrite for snowflake).
5. **Silver Load Type**: Similarly, you can select from the available load types for Silver (currently only overwrite for snowflake).
6. **Source/Bronze Watermark**: If set to None, no watermark will be established, triggering a full read from snowflake. None is recommended for Snowflake.Please provide a regular expression to identify a column to act as a watermark. If there are multiple matches, the system will select the one that appears closest to the end of the table definition in the information schema.
7. **Bronze/Silver Watermark**: If set to *None*, no watermark will be established, triggering a full read from the bronze data layer. None is recommended for Snowflake. If set to *__created_tsp*, only newly arrived records in the bronze layer will be considered when transferring data to the silver layer.
8. **Additional Options**: The 'Set Nullable' option updates each column's nullable attribute based on what's recorded in the snowflake information schema. The 'Set Primary Keys' option sets each column's metadata key attribute to 'true' if that column in snowflake is constrained by a Primary or Unique Key. The 'Get Snowflake Comment' will try and retrieve a comment from Snowflake for a similar named field, if no source comment is present.

In [0]:
import os, sys

import data.utilities.common as Commonlib
import yaml
from beertech_utils.core.logger import LogManager
from data.utilities.extractors import SnowflakeExtractor
from pyspark.sql.functions import desc, expr
from pyspark.sql.types import (
    BinaryType,
    BooleanType,
    ByteType,
    DateType,
    DecimalType,
    DoubleType,
    FloatType,
    IntegerType,
    LongType,
    ShortType,
    StringType,
    TimestampType,
)

In [0]:
OPTION_NULLABLE = "Set Nullable"

CONFIG_BASE_PATH = "../../configuration/snowflake/"
BRONZE_LOAD_TYPES = ["overwrite"]
SILVER_LOAD_TYPES = ["overwrite"]
BOOLEAN_OPTIONS = ["yes", "no"]
COMMENT_OPTIONS = ["Source, FieldName"]
BRONZE_WATERMARK_OPTIONS = ["None", ".*MOD.*TSP$", ".*MDFD.*TSP$"]
SILVER_WATERMARK_OPTIONS = ["None", "__created_tsp"]
OTHER_OPTIONS = [OPTION_NULLABLE]
MAX_ROWS_ALLOWED_OVERWRITE = 100000

# Configure the logging module
logger = LogManager.get_logger("Snowflake Configuration Generator")

#### get value from all widgets that should not cause snowflake queries to execute
# get available schemas - only if we have not retrieved it before - to force refresh detach and re-attach notebook
if "snowflake_schema" not in globals():
    snowflake_schema = "PUBLIC"
    snowflake_extractor = SnowflakeExtractor("ABI_WH", snowflake_schema)

    available_schemas_query = "select distinct table_schema as owner from INFORMATION_SCHEMA.TABLES t order by 1 asc"
    available_schemas = snowflake_extractor.query(available_schemas_query)
    available_schemas = [row.OWNER.lower() for row in available_schemas.select("OWNER").collect()]

    # display available schemas
    dbutils.widgets.dropdown("snowflake_schema", "", [""] + available_schemas, "1. Select Schema")

# subject areas available for configurations - configs will be created in this location
subject_entries = Commonlib.list_directories(CONFIG_BASE_PATH)
dbutils.widgets.dropdown("subject_area", "", [""] + sorted(subject_entries), "3. Subject Area for Configs")
subject_area = dbutils.widgets.get("subject_area")

# set load types available
dbutils.widgets.dropdown("bronze_load_type", "overwrite", BRONZE_LOAD_TYPES, "4. Bronze Load Type")
bronze_load_type = dbutils.widgets.get("bronze_load_type")
dbutils.widgets.dropdown("silver_load_type", "overwrite", SILVER_LOAD_TYPES, "5. Silver Load Type")
silver_load_type = dbutils.widgets.get("silver_load_type")

# display available options
dbutils.widgets.multiselect("other_options", "", [""] + OTHER_OPTIONS, "8. Other Options")
other_selections = dbutils.widgets.get("other_options")
other_selections = other_selections.split(",")

# source/bronze watermark logic
dbutils.widgets.combobox("bronze_watermark", "None", BRONZE_WATERMARK_OPTIONS, "6. Source/Bronze Watermark")
bronze_watermark = dbutils.widgets.get("bronze_watermark")

# bronze/silver watermark logic
dbutils.widgets.combobox("silver_watermark", "None", SILVER_WATERMARK_OPTIONS, "7. Bronze/Silver Watermark")
silver_watermark = dbutils.widgets.get("silver_watermark")

In [0]:
#### get value from widgets that should cause a table dropdown refresh
# get value from schema
snowflake_schema = dbutils.widgets.get("snowflake_schema")

# get available tables for specified schema
available_tables_query = f"select distinct table_name from INFORMATION_SCHEMA.TABLES t where table_schema = UPPER('{snowflake_schema}') order by 1 asc"

available_tables = snowflake_extractor.query(available_tables_query)

# display available tables
available_tables = [row.TABLE_NAME.lower() for row in available_tables.select("TABLE_NAME").collect()]

# truncate list to 1024 values
if len(available_tables) > 1023:
    del available_tables[1022:]
    logger.warning(
        f"The schema, {snowflake_schema}, contains more than 1024 values; Truncating to show only 1024 values if your table is not listed you will have to manually type it in the '6. Select Table' textbox."
    )

dbutils.widgets.combobox("snowflake_table", "", [""] + available_tables, "2. Select Table")

In [0]:
# get selected table
snowflake_table = dbutils.widgets.get("snowflake_table").upper()

# if blank set value to get all tables in schema
# snowflake_table = snowflake_table.upper()

logger.info(f"Tables to generate: {snowflake_table}")

# get detailed data from snowflake information schema
with open("snowflake_generator_info_schema.sql", "r") as file:
    snowflake_info_schema_query = file.read()

snowflake_info_schema_query = snowflake_info_schema_query.format(
    snowflake_schema=snowflake_schema, snowflake_table=snowflake_table
)

In [0]:
# confirm subject area is set and snowflake_table is set - prevents massive amount of files from being generated at configuration/snowflake root folder
if not subject_area:
    raise ValueError("Subject area must be set in order to continue.")

if not snowflake_table:
    raise ValueError("Select or provide table name in order to continue.")

info_schema_df = snowflake_extractor.query(snowflake_info_schema_query)

# cast to correct data types
info_schema_df = info_schema_df.withColumn("DATA_SCALE", info_schema_df["DATA_SCALE"].cast("integer"))
info_schema_df = info_schema_df.withColumn("DATA_PRECISION", info_schema_df["DATA_PRECISION"].cast("integer"))

In [0]:
#### helpers
# snowflake data types mapping for pyspark
snowflake_to_pyspark = {
    "NUMBER": DecimalType(38, 0),
    "FLOAT": FloatType(),
    "DOUBLE": DoubleType(),
    "VARCHAR": StringType(),
    "TEXT": StringType(),
    "VARIANT": StringType(),
    "CHAR": StringType(),
    "BOOLEAN": BooleanType(),
    "DATE": DateType(),
    "TIME": TimestampType(),
    "DATETIME": TimestampType(),
    "TIMESTAMP": TimestampType(),
    "TIMESTAMP_LTZ": TimestampType(),
    "TIMESTAMP_NTZ": TimestampType(),
    "TIMESTAMP_TZ": TimestampType(),
    "BINARY": BinaryType(),
    "VARBINARY": BinaryType(),
}

# metadata columns - currently used in silver for delta loads from bronze
default_created_tsp = {
    "name": "__created_tsp",
    "type": "timestamp",
    "nullable": False,
    "metadata": {"comment": "Timestamp indicating the time at which the record was originally created in delta table."},
}

default_source = {
    "name": "__source",
    "type": "string",
    "nullable": False,
    "metadata": {"comment": "Metadata field indicating the source of the record."},
}


# generates a schema field
def generate_field(row, other_options):
    """
    This function generates a dictionary representing a schema field.

    The function takes a Row object as input, which contains metadata about a column
    (like its name, type, nullable property, etc.) and returns a dictionary
    that represents the schema field for that column.

    Args:
        row (pyspark.sql.Row): A Row object containing metadata about a column.

    Returns:
        dict: A dictionary representing the schema field for the column.
    """
    col_data_precision = row.DATA_PRECISION
    col_data_scale = row.DATA_SCALE
    col_comment = row.COLUMN_COMMENTS.replace("\n", " ") if row.COLUMN_COMMENTS else row.COLUMN_NAME.lower()

    # data type handling for special cases
    if row.DATA_TYPE == "NUMBER":
        field = {"name": row.COLUMN_NAME.lower(), "type": f"decimal({col_data_precision}, {col_data_scale})"}
    else:
        field = {"name": row.COLUMN_NAME.lower(), "type": snowflake_to_pyspark[row.DATA_TYPE].typeName()}

    # determine if field is nullable - only set if false - default is true
    if row.NULLABLE == "NO" and OPTION_NULLABLE in other_options:
        field["nullable"] = False

    # generate metadata dict
    field_metadata = {"metadata": {"comment": col_comment}}

    return {**field, **field_metadata}


# generates a configuration for an entire table
def process_table(table_name, table_df, bronze_load_type=bronze_load_type, other_options=None):
    """
    This function generates a configuration dictionary for a table.

    The function takes a table name, a DataFrame containing metadata about the table,
    and a load type as input. It generates a configuration dictionary for the table,
    which includes its schema and other properties.

    The function also handles several special cases, such as large data sets and
    watermarks for bronze and silver data.

    Args:
        table_name (str): The name of the table.
        table_df (pyspark.sql.DataFrame): A DataFrame containing metadata about the table.
        bronze_load_type (str): The load type for the bronze data.

    Returns:
        dict: A dictionary representing the configuration for the table.
    """
    schema = []
    config = {}
    bronze_watermark_col = ""

    # Get the first record for current table_name
    first_record = table_df.first()

    # start building config for table
    config = {
        "version": "1.0.0",
        "name": table_name.lower(),
        "schema": "static",
        "catalog": "external",
        "description": first_record.TABLE_COMMENTS if first_record.TABLE_COMMENTS else table_name.lower(),
        "source": {"schema": first_record.OWNER.lower(), "name": table_name.lower()},
        "bronze": {"load_type": bronze_load_type},
        "silver": {"load_type": silver_load_type},
    }

    # determine bronze watermark
    if bronze_watermark != "None":
        bronze_watermark_df = table_df.filter(
            (table_df.DATA_TYPE == "DATE") | (table_df.DATA_TYPE == "TIMESTAMP")
        ).filter(table_df["COLUMN_NAME"].rlike(bronze_watermark))
        # get best match for bronze watermark
        if bronze_watermark_df.count() > 0:
            # sort by column id - we want the match closes to the end
            bronze_watermark_df = bronze_watermark_df.sort(desc("COLUMN_ID"))
            bronze_watermark_df = bronze_watermark_df.first()
            bronze_watermark_col = bronze_watermark_df["COLUMN_NAME"]
            config["bronze"]["watermark"] = bronze_watermark_col.lower()
        else:
            config["bronze"]["watermark"] = "auto_find_failed"

    # set silver watermark
    if silver_watermark == "Same as bronze" and bronze_watermark_col:
        config["silver"]["watermark"] = bronze_watermark_col.lower()
    elif silver_watermark == "__created_tsp":
        config["silver"]["watermark"] = "__created_tsp"

    # loop through every row/column to genrate the schema field
    for row in table_df.collect():
        schema.append(generate_field(row, other_options))

    # add default columns
    schema.append(default_created_tsp)
    schema.append(default_source)

    config["silver"]["schema"] = schema

    return config


#### generate configurations
# get tables that need configurations
table_names = [row.TABLE_NAME for row in info_schema_df.select("TABLE_NAME").distinct().sort("TABLE_NAME").collect()]

# loop through all the tables and generate config file in proper folder
for table_name in table_names:
    logger.info(f"Processing table: {table_name}")

    table_df = info_schema_df.filter(info_schema_df.TABLE_NAME == table_name).sort(info_schema_df.COLUMN_ID.asc())

    config = process_table(table_name, table_df, other_options=other_selections)
    config_write_path = f"{CONFIG_BASE_PATH}{subject_area}"
    config_file = f"{table_name.lower()}.yml"

    with open(os.path.join(config_write_path, config_file), "w") as outfile:
        yaml.dump(config, outfile, indent=2, sort_keys=False)